# Higgs → γγ classifier: logistic regression vs XGBoost (BDT)

**Setup**
- Background for training: preselected **data A**. Signal for training: Higgs → γγ **MC** (50% of events; the other 50% is held out for testing).
- Test: **data B, C, D** as background, held-out MC as signal.
- The diphoton mass `m_γγ` is **not** a training feature (set `USE_MASS = True` to test the effect), so the classifier cannot just learn the 125 GeV bump.
- Evaluation: ROC/AUC, then the thing that matters for physics: Asimov significance of the Higgs bump in a 120–130 GeV window after a cut on the classifier score.

In [ ]:
import numpy as np
import awkward as ak
import uproot
import matplotlib.pyplot as plt
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, roc_curve
from xgboost import XGBClassifier

SEED = 42
rng = np.random.default_rng(SEED)

## 1. Config

In [ ]:
DATA_DIR  = Path("../data/GamGam")            # adjust: folder with Data/ and MC/
FIG_DIR   = Path("../results/figures")
SCORE_DIR = Path("../results/scores")
FIG_DIR.mkdir(parents=True, exist_ok=True)
SCORE_DIR.mkdir(parents=True, exist_ok=True)

LUMI = {"A": 0.5, "B": 1.9, "C": 2.9, "D": 4.7}   # fb^-1
TRAIN_PERIOD = "A"
TEST_PERIODS = ["B", "C", "D"]
LUMI_TEST = sum(LUMI[p] for p in TEST_PERIODS)

DATA_FILES = {p: DATA_DIR / "Data" / f"data_{p}.GamGam.root" for p in LUMI}
MC_FILES   = sorted((DATA_DIR / "MC").glob("mc_*.GamGam.root"))
TREE = "mini"

USE_MASS = False   # keep False: mass would let the model sculpt the bump
FEATURES = ["pt1", "pt2", "abs_eta1", "abs_eta2", "dphi", "deta", "dR", "pt_gg", "abs_y_gg"]
if USE_MASS:
    FEATURES.append("mgg")

# significance study
MASS_BINS = np.arange(105, 161, 1.0)
CENTERS = 0.5 * (MASS_BINS[1:] + MASS_BINS[:-1])
WINDOW = (120.0, 130.0)

print("MC files:", [f.name for f in MC_FILES])

## 2. Load + preselect

Same cuts as the tutorial (trigger, tight ID, pT, isolation, calorimeter crack). If you already have these in `src/preselection.py`, swap `process_chunk` for your own version and keep the feature block.

In [ ]:
BRANCHES = ["trigP", "photon_pt", "photon_eta", "photon_phi", "photon_e",
            "photon_isTightID", "photon_ptcone30", "photon_etcone20"]
MC_BRANCHES = ["mcWeight", "scaleFactor_PILEUP", "scaleFactor_PHOTON",
               "scaleFactor_PhotonTRIGGER", "XSection", "SumWeights"]

def in_crack(eta):
    return (np.abs(eta) > 1.37) & (np.abs(eta) < 1.52)

def process_chunk(a, is_mc, lumi):
    a = a[ak.num(a["photon_pt"]) >= 2]
    two = lambda name: (ak.to_numpy(a[name][:, 0]), ak.to_numpy(a[name][:, 1]))

    pt1, pt2 = [x / 1000 for x in two("photon_pt")]   # GeV
    eta1, eta2 = two("photon_eta")
    phi1, phi2 = two("photon_phi")
    e1, e2 = [x / 1000 for x in two("photon_e")]
    tight1, tight2 = two("photon_isTightID")
    cone30_1, cone30_2 = two("photon_ptcone30")
    et20_1, et20_2 = two("photon_etcone20")
    trig = ak.to_numpy(a["trigP"])

    sel = (trig & tight1 & tight2
           & (pt1 > 40) & (pt2 > 30)
           & (cone30_1 / (pt1 * 1000) < 0.065) & (cone30_2 / (pt2 * 1000) < 0.065)
           & (et20_1 / (pt1 * 1000) < 0.065) & (et20_2 / (pt2 * 1000) < 0.065)
           & ~in_crack(eta1) & ~in_crack(eta2))

    pt1, pt2, eta1, eta2, phi1, phi2, e1, e2 = [
        x[sel] for x in (pt1, pt2, eta1, eta2, phi1, phi2, e1, e2)]

    # kinematics
    dphi = np.abs(phi1 - phi2)
    dphi = np.where(dphi > np.pi, 2 * np.pi - dphi, dphi)
    deta = np.abs(eta1 - eta2)
    mgg = np.sqrt(2 * pt1 * pt2 * (np.cosh(eta1 - eta2) - np.cos(phi1 - phi2)))
    px = pt1 * np.cos(phi1) + pt2 * np.cos(phi2)
    py = pt1 * np.sin(phi1) + pt2 * np.sin(phi2)
    pz = pt1 * np.sinh(eta1) + pt2 * np.sinh(eta2)
    E = e1 + e2
    y_gg = 0.5 * np.log((E + pz) / (E - pz))

    out = {
        "pt1": pt1, "pt2": pt2,
        "abs_eta1": np.abs(eta1), "abs_eta2": np.abs(eta2),
        "dphi": dphi, "deta": deta, "dR": np.hypot(deta, dphi),
        "pt_gg": np.hypot(px, py), "abs_y_gg": np.abs(y_gg),
        "mgg": mgg,
    }

    if is_mc:
        s = lambda n: ak.to_numpy(a[n])[sel]
        out["w"] = (s("mcWeight") * s("scaleFactor_PILEUP") * s("scaleFactor_PHOTON")
                    * s("scaleFactor_PhotonTRIGGER")
                    * s("XSection") * 1000 * lumi / s("SumWeights"))
    else:
        out["w"] = np.ones(sel.sum())
    return out

def load(path, is_mc, lumi=1.0):
    branches = BRANCHES + (MC_BRANCHES if is_mc else [])
    chunks = [process_chunk(a, is_mc, lumi)
              for a in uproot.iterate(f"{path}:{TREE}", branches, step_size="100 MB", library="ak")]
    return {k: np.concatenate([c[k] for c in chunks]) for k in chunks[0]}

def concat(dicts):
    return {k: np.concatenate([d[k] for d in dicts]) for k in dicts[0]}

def to_matrix(d):
    return np.column_stack([d[f] for f in FEATURES])

In [ ]:
bkg_train_raw = load(DATA_FILES[TRAIN_PERIOD], is_mc=False)
data_test = {p: load(DATA_FILES[p], is_mc=False) for p in TEST_PERIODS}
data_bcd = concat(list(data_test.values()))
sig_raw = concat([load(f, is_mc=True, lumi=LUMI_TEST) for f in MC_FILES])   # MC scaled to B+C+D lumi

print(f"data {TRAIN_PERIOD} (bkg, train): {len(bkg_train_raw['w']):>9}")
for p, d in data_test.items():
    print(f"data {p} (test):          {len(d['w']):>9}")
print(f"MC signal (raw events):   {len(sig_raw['w']):>9}   expected yield @ {LUMI_TEST:.1f} fb^-1: {sig_raw['w'].sum():.1f}")

## 3. Train/validation/test split

- Signal MC: 50% train (of which 20% validation for early stopping), 50% test.
- Background: data A → train/validation. Data B, C, D → test only.
- Training weights: each class is rescaled to the same total weight (MC weights are tiny compared to counting data events).

In [ ]:
X_sig, w_sig = to_matrix(sig_raw), sig_raw["w"]
X_bkg_A = to_matrix(bkg_train_raw)
X_bcd = to_matrix(data_bcd)

i_tr, i_te = train_test_split(np.arange(len(X_sig)), test_size=0.5, random_state=SEED)
i_tr, i_va = train_test_split(i_tr, test_size=0.2, random_state=SEED)
j_tr, j_va = train_test_split(np.arange(len(X_bkg_A)), test_size=0.2, random_state=SEED)

def make_set(Xs, ws, Xb, wb):
    X = np.vstack([Xs, Xb])
    y = np.r_[np.ones(len(Xs)), np.zeros(len(Xb))]
    w = np.r_[ws, wb]
    return X, y, w

def balanced(y, w):
    # |weights|, each class normalised to the same total
    w = np.abs(w).astype(float).copy()
    for c in (0, 1):
        m = y == c
        w[m] *= 0.5 * len(y) / w[m].sum()
    return w

X_tr, y_tr, w_tr = make_set(X_sig[i_tr], w_sig[i_tr], X_bkg_A[j_tr], np.ones(len(j_tr)))
X_va, y_va, w_va = make_set(X_sig[i_va], w_sig[i_va], X_bkg_A[j_va], np.ones(len(j_va)))
X_te, y_te, w_te = make_set(X_sig[i_te], w_sig[i_te], X_bcd, np.ones(len(X_bcd)))

w_tr_b, w_va_b, w_te_b = balanced(y_tr, w_tr), balanced(y_va, w_va), balanced(y_te, w_te)
print("train:", X_tr.shape, " val:", X_va.shape, " test:", X_te.shape)

## 4. Train both models

In [ ]:
logreg = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
logreg.fit(X_tr, y_tr, logisticregression__sample_weight=w_tr_b)

xgb = XGBClassifier(
    n_estimators=500, max_depth=4, learning_rate=0.1,
    subsample=0.8, colsample_bytree=0.8,
    eval_metric="auc", early_stopping_rounds=30,
    n_jobs=-1, random_state=SEED,
)
xgb.fit(X_tr, y_tr, sample_weight=w_tr_b,
        eval_set=[(X_va, y_va)], sample_weight_eval_set=[w_va_b], verbose=False)
print("XGBoost best iteration:", xgb.best_iteration)

models = {"Logistic regression": logreg, "XGBoost (BDT)": xgb}
scores_te = {name: m.predict_proba(X_te)[:, 1] for name, m in models.items()}

## 5. Classifier performance (held-out MC vs data B+C+D)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4))

for name, s in scores_te.items():
    auc = roc_auc_score(y_te, s, sample_weight=w_te_b)
    fpr, tpr, _ = roc_curve(y_te, s, sample_weight=w_te_b)
    ax[0].plot(fpr, tpr, label=f"{name} (AUC = {auc:.3f})")
ax[0].plot([0, 1], [0, 1], "k--", lw=0.8)
ax[0].set(xlabel="Background efficiency (data B+C+D)", ylabel="Signal efficiency (MC)", title="ROC")
ax[0].legend()

s = scores_te["XGBoost (BDT)"]
bins = np.linspace(0, 1, 41)
ax[1].hist(s[y_te == 0], bins, density=True, histtype="step", label="data B+C+D (bkg)")
ax[1].hist(s[y_te == 1], bins, density=True, histtype="step", weights=np.abs(w_te[y_te == 1]), label="MC (signal)")
ax[1].set(xlabel="XGBoost score", ylabel="Normalised", title="XGBoost score distribution")
ax[1].legend()

plt.tight_layout()
plt.savefig(FIG_DIR / "classifier_roc_scores.png", dpi=150)
plt.show()

imp = xgb.feature_importances_
order = np.argsort(imp)
plt.figure(figsize=(5, 3.5))
plt.barh(np.array(FEATURES)[order], imp[order])
plt.xlabel("XGBoost feature importance (gain)")
plt.tight_layout()
plt.savefig(FIG_DIR / "xgb_feature_importance.png", dpi=150)
plt.show()

## 6. Does the score sculpt the mass spectrum?

If the correlation between the score and `m_γγ` in the background data is large, a cut on the score will distort the background shape and make the bump fit unreliable.

In [ ]:
mgg_bcd = data_bcd["mgg"]
for name, m in models.items():
    sc = m.predict_proba(X_bcd)[:, 1]
    sel = (mgg_bcd > 105) & (mgg_bcd < 160)
    print(f"{name:22s} corr(score, m_gg) on data B+C+D: {np.corrcoef(sc[sel], mgg_bcd[sel])[0, 1]:+.3f}")

## 7. Significance of the Higgs bump vs a cut on the score

- Background in 120-130 GeV: polynomial (degree 3) fitted to the sidebands of data B+C+D after the cut.
- Signal: held-out MC in the window, weighted to the B+C+D luminosity (x2 because only half of the MC is in the test set).
- Significance: Asimov formula, `Z = sqrt(2((S+B) ln(1+S/B) - S))`.
- Cuts are scanned in signal efficiency.

In [ ]:
in_win = (CENTERS > WINDOW[0]) & (CENTERS < WINDOW[1])

def background_in_window(mgg, deg=3):
    n, _ = np.histogram(mgg, bins=MASS_BINS)
    x = (CENTERS - 125) / 25
    side = ~in_win
    coef = np.polyfit(x[side], n[side], deg, w=1 / np.sqrt(np.maximum(n[side], 1)))
    fit = np.polyval(coef, x)
    return max(fit[in_win].sum(), 1e-9), n, fit

def asimov_Z(S, B):
    return np.sqrt(2 * ((S + B) * np.log(1 + S / B) - S))

def cut_for_efficiency(scores, w, eff):
    order = np.argsort(scores)[::-1]
    cum = np.cumsum(np.abs(w[order])) / np.abs(w).sum()
    return scores[order][min(np.searchsorted(cum, eff), len(order) - 1)]

mgg_sig_te = sig_raw["mgg"][i_te]
w_sig_te = 2 * w_sig[i_te]                       # x2: only half of the MC is used for testing
sig_in_win = (mgg_sig_te >= WINDOW[0]) & (mgg_sig_te < WINDOW[1])

def significance_for_cut(score_sig, score_bkg, cut):
    S = w_sig_te[(score_sig >= cut) & sig_in_win].sum()
    B, _, _ = background_in_window(mgg_bcd[score_bkg >= cut])
    return S, B, asimov_Z(S, B)

# baseline: no classifier cut
S0, B0, Z0 = significance_for_cut(np.ones(len(w_sig_te)), np.ones(len(mgg_bcd)), 0.5)
print(f"No cut:  S = {S0:.1f}, B = {B0:.0f}, Z = {Z0:.2f}")

effs = np.r_[np.arange(0.2, 1.0, 0.05), 1.0]
results = {}
for name, m in models.items():
    s_sig = m.predict_proba(X_sig[i_te])[:, 1]
    s_bkg = m.predict_proba(X_bcd)[:, 1]
    rows = []
    for eff in effs:
        cut = cut_for_efficiency(s_sig, w_sig_te, eff)
        S, B, Z = significance_for_cut(s_sig, s_bkg, cut)
        rows.append((eff, cut, S, B, Z))
    results[name] = np.array(rows)
    best = results[name][np.argmax(results[name][:, 4])]
    print(f"{name:22s} best Z = {best[4]:.2f} at signal eff {best[0]:.2f} (cut {best[1]:.3f}): S = {best[2]:.1f}, B = {best[3]:.0f}")

In [ ]:
plt.figure(figsize=(6, 4))
for name, r in results.items():
    plt.plot(r[:, 0], r[:, 4], "o-", label=name)
plt.axhline(Z0, color="k", ls="--", lw=0.8, label="no classifier cut")
plt.xlabel("Signal efficiency of the score cut")
plt.ylabel("Asimov significance (120-130 GeV)")
plt.legend()
plt.tight_layout()
plt.savefig(FIG_DIR / "significance_vs_efficiency.png", dpi=150)
plt.show()

# m_gg spectrum before/after the best XGBoost cut
best = results["XGBoost (BDT)"][np.argmax(results["XGBoost (BDT)"][:, 4])]
s_bkg = xgb.predict_proba(X_bcd)[:, 1]
fig, ax = plt.subplots(1, 2, figsize=(11, 4), sharex=True)
for a_, mask, title in [(ax[0], np.ones(len(mgg_bcd), bool), "Preselection only"),
                        (ax[1], s_bkg >= best[1], f"XGBoost score >= {best[1]:.2f}")]:
    B, n, fit = background_in_window(mgg_bcd[mask])
    a_.errorbar(CENTERS, n, yerr=np.sqrt(n), fmt="k.", label="data B+C+D")
    a_.plot(CENTERS, fit, "r-", label="sideband poly fit")
    a_.axvspan(*WINDOW, color="C0", alpha=0.1)
    a_.set(xlabel=r"$m_{\gamma\gamma}$ [GeV]", ylabel="Events / GeV", title=title)
    a_.legend()
plt.tight_layout()
plt.savefig(FIG_DIR / "mgg_before_after_cut.png", dpi=150)
plt.show()

## 8. Save scores

In [ ]:
np.savez(
    SCORE_DIR / "scores_logreg_xgb.npz",
    features=np.array(FEATURES),
    mgg_bcd=mgg_bcd,
    xgb_bcd=xgb.predict_proba(X_bcd)[:, 1],
    logreg_bcd=logreg.predict_proba(X_bcd)[:, 1],
    mgg_sig_test=mgg_sig_te, w_sig_test=w_sig_te,
    xgb_sig_test=xgb.predict_proba(X_sig[i_te])[:, 1],
    logreg_sig_test=logreg.predict_proba(X_sig[i_te])[:, 1],
)
print("saved")

## Notes

- The best cut is chosen on the same test data it is evaluated on, so the quoted significance is slightly optimistic. Fine for comparing the two models; for a final number, choose the cut on the validation set.
- Sanity-check the MC yield (printed in section 2) against the tutorial. If it is far off, the cross-section weighting needs adjusting.
- Data A also contains a few Higgs events. That contamination is negligible compared to the background.
- To test what the mass does, set `USE_MASS = True` and re-run: you should see the significance jump and the sculpting correlation in section 6 get large.